# 05 — Preparação dos dados de ML

Código espelhado de `RETREINO/03_preparation.py`. `tbl_ml` foi substituída por `desloc.gold.transacoes_ml`; `distancia` por `distancia_lalo`. O pipeline do professor mostra `drop` de `periodo` e `MinMaxScaler`.

In [ ]:
df = spark.table("desloc.gold.transacoes_ml").toPandas()

# Não recomendado
# df.drop(columns=["periodo"], inplace=True)

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["vlr_pago"])
Y = df["vlr_pago"]

X_train, X_test, y_train, y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42
)

In [ ]:
colunas_minmax = [
    'dia',
    'dia_semana',
    'mes',
    'hora',
    'distancia_lalo',
    'latitude_origem',
    'longitude_origem',
    'latitude_destino',
    'longitude_destino'
]

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler

# Desabilitar o MLFLOW automatico do Databricks
import mlflow
mlflow.autolog(disable=True)

# Definir os processadores
preprocessador = ColumnTransformer(
    transformers=[
        ('drop', 'drop', ['periodo']),
        ('minmax', MinMaxScaler(), colunas_minmax)
    ],
    remainder='passthrough'  # mantém as outras colunas sem alteração
)

# Combinar os processadores + pipeline
pipeline = Pipeline(
    steps=[
        ('preprocessamento', preprocessador),
    ]
)

# Gerar o objeto do pipeline
pipeline.fit(X_train, y_train)

In [ ]:
X_train.head(5)

In [ ]:
pipeline

In [ ]:
import pandas as pd
pd.DataFrame(pipeline.transform(X_train))

In [ ]:
import pandas as pd

pd.DataFrame( pipeline.transform(X_train) ).describe().T

## Explicabilidade (XAI) e limite de negócio
O roteiro prepara os dados para a aula de modelagem. As features de calendário ainda usam a hora de entrega; não são entradas prontas para cotação antes da corrida.